#PySpark Practice Notebook

In [1]:
#Source Data
#Create a csv file for the following data
"""
product_id,product,country,sales
1,Product A,USA,100
2,Product B,USA,80
3,Product C,USA,30
1,Product A,Canada,60
2,Product B,Canada,40
4,Product D,UK,50
5,Product E,UK,20
1,Product A,Germany,70
3,Product C,Germany,90
4,Product D,Germany,40


"""

'\nproduct_id,product,country,sales\n1,Product A,USA,100\n2,Product B,USA,80\n3,Product C,USA,30\n1,Product A,Canada,60\n2,Product B,Canada,40\n4,Product D,UK,50\n5,Product E,UK,20\n1,Product A,Germany,70\n3,Product C,Germany,90\n4,Product D,Germany,40\n\n\n'

In [2]:
#Import Pyspark & other necessary functions
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
#Create SparkSession for app "Sales Data Analysis"
spark=SparkSession.builder.appName("Sales Data Analysis").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 04:57:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#Create a pyspark DataFrame from the csv file on local storage
#Use StructType Declaration for column creation
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("country", StringType(), True),
    StructField("sales", IntegerType(), True)
])



In [5]:
#Verify schema for the newly created file
df = spark.read.csv("data.csv",header=True,schema=schema)
df.show()

+----------+---------+-------+-----+
|product_id|  product|country|sales|
+----------+---------+-------+-----+
|         1|Product A|    USA|  100|
|         2|Product B|    USA|   80|
|         3|Product C|    USA|   30|
|         1|Product A| Canada|   60|
|         2|Product B| Canada|   40|
|         4|Product D|     UK|   50|
|         5|Product E|     UK|   20|
|         1|Product A|Germany|   70|
|         3|Product C|Germany|   90|
|         4|Product D|Germany|   40|
+----------+---------+-------+-----+



In [6]:
#Display all entries for country "Germany"
df.filter("country='Germany'").show()

+----------+---------+-------+-----+
|product_id|  product|country|sales|
+----------+---------+-------+-----+
|         1|Product A|Germany|   70|
|         3|Product C|Germany|   90|
|         4|Product D|Germany|   40|
+----------+---------+-------+-----+



In [7]:
#Find the total number of products
df.groupby("product").count().show()

+---------+-----+
|  product|count|
+---------+-----+
|Product A|    3|
|Product B|    2|
|Product C|    2|
|Product D|    2|
|Product E|    1|
+---------+-----+



In [8]:
#Find top 3 products
df.groupby("product").agg(F.sum("Sales").alias("Sum_sales")).orderBy(F.desc("Sum_sales")).show(3)

+---------+---------+
|  product|Sum_sales|
+---------+---------+
|Product A|      230|
|Product B|      120|
|Product C|      120|
+---------+---------+
only showing top 3 rows


In [9]:
#Calculate total sales
df1=df.select(F.sum("sales").alias("Total_sales")).collect()[0][0]
print(df1)

580


In [10]:
#Find the market share of all products individually
df.groupby("product").agg((F.sum("sales")*100/df1).alias("Market Share")).show()

+---------+------------------+
|  product|      Market Share|
+---------+------------------+
|Product A|  39.6551724137931|
|Product B|20.689655172413794|
|Product C|20.689655172413794|
|Product D|15.517241379310345|
|Product E|3.4482758620689653|
+---------+------------------+



In [11]:
df.createOrReplaceTempView("products")

In [13]:
a=spark.sql("""select * from products""")
a

DataFrame[product_id: int, product: string, country: string, sales: int]

In [14]:
type(a)

pyspark.sql.classic.dataframe.DataFrame

In [15]:
a.show(3)

+----------+---------+-------+-----+
|product_id|  product|country|sales|
+----------+---------+-------+-----+
|         1|Product A|    USA|  100|
|         2|Product B|    USA|   80|
|         3|Product C|    USA|   30|
+----------+---------+-------+-----+
only showing top 3 rows


In [19]:
spark.sql("""

select sum(sales) as Total_sales from products

""").show()

+-----------+
|Total_sales|
+-----------+
|        580|
+-----------+



In [21]:
cat_df = spark.read.csv("product_cat.csv",header=True,inferSchema=True)

cat_df.show()

+----------+-----------+-----+
|product_id|   category|price|
+----------+-----------+-----+
|         1|Electronics|  500|
|         2|Electronics|  750|
|         3|  Furniture|  300|
|         4|  Furniture|  450|
|         6|       Toys|  100|
+----------+-----------+-----+



In [23]:
jdf=df.join(cat_df)

In [24]:
jdf.show()

+----------+---------+-------+-----+----------+-----------+-----+
|product_id|  product|country|sales|product_id|   category|price|
+----------+---------+-------+-----+----------+-----------+-----+
|         1|Product A|    USA|  100|         1|Electronics|  500|
|         1|Product A|    USA|  100|         2|Electronics|  750|
|         1|Product A|    USA|  100|         3|  Furniture|  300|
|         1|Product A|    USA|  100|         4|  Furniture|  450|
|         1|Product A|    USA|  100|         6|       Toys|  100|
|         2|Product B|    USA|   80|         1|Electronics|  500|
|         2|Product B|    USA|   80|         2|Electronics|  750|
|         2|Product B|    USA|   80|         3|  Furniture|  300|
|         2|Product B|    USA|   80|         4|  Furniture|  450|
|         2|Product B|    USA|   80|         6|       Toys|  100|
|         3|Product C|    USA|   30|         1|Electronics|  500|
|         3|Product C|    USA|   30|         2|Electronics|  750|
|         

In [25]:
jdf.count()

50

In [27]:
jdf2=df.join(cat_df,on="product_id",how="inner")

In [29]:
jdf2.show()

+----------+---------+-------+-----+-----------+-----+
|product_id|  product|country|sales|   category|price|
+----------+---------+-------+-----+-----------+-----+
|         1|Product A|    USA|  100|Electronics|  500|
|         2|Product B|    USA|   80|Electronics|  750|
|         3|Product C|    USA|   30|  Furniture|  300|
|         1|Product A| Canada|   60|Electronics|  500|
|         2|Product B| Canada|   40|Electronics|  750|
|         4|Product D|     UK|   50|  Furniture|  450|
|         1|Product A|Germany|   70|Electronics|  500|
|         3|Product C|Germany|   90|  Furniture|  300|
|         4|Product D|Germany|   40|  Furniture|  450|
+----------+---------+-------+-----+-----------+-----+

